# Industrial Pump Condition Classification

**AI Services MS2 | Supplementary Classification Practice | 60 marks | 20 tests**

A maintenance team must classify pump inspections as **Healthy**, **Inspect**, or **Urgent**. Complete the independent functions, compare two classifiers on held-back data, save the winning fitted pipeline and score queued inspections without retraining.

**Input files:** `pump_train.csv`, `pump_test.csv`, `pump_predict.csv`. Keep all three beside this notebook. The model/report outputs are generated only when the workflow cells are run.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

TARGET = "condition_label"
CLASS_NAMES = ["Healthy", "Inspect", "Urgent"]
raw_numeric_cols = [
    "inlet_pressure_bar", "outlet_pressure_bar", "flow_volume_m3",
    "runtime_hours", "motor_temperature_c", "ambient_temperature_c",
    "vibration_mm_s", "operating_load_pct"
]
category_cols = ["pump_type", "maintenance_zone", "shift_code"]
numeric_features = raw_numeric_cols + [
    "inspection_hour", "pressure_gain_bar", "flow_rate_m3h", "temperature_rise_c"
]
features = numeric_features + category_cols

## Function A1 - Common loader

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    """Read and clean inspection records; engineer four reusable features."""
    df = pd.read_csv(filename, dtype={"inspection_id": "string"})

    df["inspection_id"]=df["inspection_id"].str.strip().replace("",pd.NA)
    df = df.dropna(subset=["inspection_id"])
    df = df.drop_duplicates("inspection_id", keep="first").copy()

    df["inspected_at"] = pd.to_datetime(df["inspected_at"],format="mixed",dayfirst=True,errors="coerce")

    df = df.dropna(subset="inspected_at")

    df["inspection_hour"] = df["inspected_at"].dt.hour

    for col in raw_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df[raw_numeric_cols] = df[raw_numeric_cols].replace([np.inf, -np.inf], np.nan)

    for col in category_cols:
        df[col] = df[col].astype("string").str.strip().str.lower().replace("", pd.NA).fillna("unknown")


    df["shift_code"] = df["shift_code"].str.replace(r"\.0+$", "", regex=True).astype("string")


    df["pump_type"] = df["pump_type"]
    df["maintenance_zone"] = df["maintenance_zone"]

    df["pressure_gain_bar"] = (df["outlet_pressure_bar"] - df["inlet_pressure_bar"])
    df["temperature_rise_c"] = (df["motor_temperature_c"] - df["ambient_temperature_c"])

    runtime = df["runtime_hours"].where(df["runtime_hours"] > 0)
    
    df["flow_rate_m3h"] = df["flow_volume_m3"] / runtime

    df["flow_rate_m3h"] = df["flow_rate_m3h"].replace([np.inf, -np.inf], np.nan)


    return df

## Function A2 - Reusable classification pipeline

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def build_pipeline(algorithm: BaseEstimator) -> Pipeline:
    """Create an unfitted model with numeric and categorical preprocessing."""
    
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("category", OneHotEncoder(handle_unknown="ignore"), category_cols)
    ])
    
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", algorithm)
    ])

## Function 1 - Prepare labeled datasets

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Prepare supervised predictors and stable three-class integer targets."""
    
    train_df = load_data(train_file)
    test_df = load_data(test_file)
    
    encoder = LabelEncoder().fit(CLASS_NAMES)

    train_df[TARGET] = train_df[TARGET].astype("string").str.strip().str.title()
    test_df[TARGET] = test_df[TARGET].astype("string").str.strip().str.title()

    train_df = train_df[train_df[TARGET].isin(CLASS_NAMES)]
    test_df = test_df[test_df[TARGET].isin(CLASS_NAMES)]

    # df.drop(df.index[~df[TARGET].isin(CLASS_NAMES)], inplace=True)

    X_train = train_df[features]
    X_test = test_df[features]

    y_train = pd.Series(encoder.transform(train_df[TARGET]), index=train_df.index, name=TARGET)
    y_test = pd.Series(encoder.transform(test_df[TARGET]), index=test_df.index, name=TARGET )
    
    return X_train, X_test, y_train, y_test

## Function 2 - Train candidate classifiers

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    """Fit two complete and independent classification pipelines."""
    
    models = {
        "logistic_regression": LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=42
        ),
        
        "random_forest": RandomForestClassifier(
            n_estimators=150, max_depth=12, min_samples_leaf=2,
            class_weight="balanced", random_state=42, n_jobs=-1
        )
    }

    trained_models = {}
    
    for name, algorithm in models.items():
        trained_models[name] = build_pipeline(algorithm).fit(X_train, y_train)
    return trained_models

## Function 3 - Evaluate and compare models

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def evaluate_models(
    trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series ) -> pd.DataFrame:
    """Rank fitted models by macro F1, accuracy, then model name."""
    results = []
    
    for name, pipeline in trained_models.items():
        predictions = pipeline.predict(X_test)
        results.append({
            "model_name": name,
            "accuracy": accuracy_score(y_test, predictions),
            "macro_f1": f1_score(
                y_test, predictions, average="macro",
                labels=[0, 1, 2], zero_division=0
            ),
            "weighted_f1": f1_score(
                y_test, predictions, average="weighted",
                labels=[0, 1, 2], zero_division=0
            )
        })
    comparison = pd.DataFrame(results)
    return comparison.sort_values(
        ["macro_f1", "accuracy", "model_name"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

## Function 4 - Select best model

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def select_best_model(
    comparison: pd.DataFrame, trained_models: dict[str, Pipeline]
) -> tuple[str, Pipeline]:
    """Return the selected name and the original fitted pipeline."""
    best_name = comparison.iloc[0]["model_name"]
    best_model = trained_models[best_name]
    return best_name, best_model

## Function 5 - Predict pump condition

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def predict_condition(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    """Convert predicted integer classes into readable condition labels."""
    X = df[features]
    predictions = np.asarray(model.predict(X), dtype=int)
    encoder = LabelEncoder().fit(CLASS_NAMES)
    labels = encoder.inverse_transform(predictions)
    urgent_code = int(encoder.transform(["Urgent"])[0])
    urgent_index = list(model.classes_).index(urgent_code)
    
    probabilities = np.round(
        model.predict_proba(X)[:, urgent_index], 4
    )
    return pd.DataFrame({
        "inspection_id": df["inspection_id"].to_numpy(),
        "predicted_condition": labels,
        "urgent_probability": probabilities
    })

## Training workflow

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    """Train candidates, rank by macro F1, persist the best model and reports."""
    X_train, X_test, y_train, y_test = prepare_datasets(
        "pump_train.csv", "pump_test.csv"
    )
    trained_models = train_models(X_train, y_train)
    comparison = evaluate_models(trained_models, X_test, y_test)
    best_name, best_model = select_best_model(comparison, trained_models)

    joblib.dump(best_model, "best_model.pkl")
    comparison.to_csv("model_comparison.csv", index=False)

    y_pred = best_model.predict(X_test)

    matrix = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])

    pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        "confusion_matrix.csv", index_label="actual_condition"
    )
    # print("Selected model:", best_name)
    # print(comparison.round(4).to_string(index=False))
    return best_name, comparison

## Prediction workflow

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def predict() -> pd.DataFrame:
    """Reload the selected pipeline and predict queued conditions, without fitting."""
    df = load_data("pump_predict.csv")
    model = joblib.load("best_model.pkl")
    result = predict_condition(model, df)
    result.to_csv("pump_predictions.csv", index=False)
    print("Predictions created:", len(result))
    print(result.head().to_string(index=False))
    return result

## Run the workflows

Run the **training** cell first. You may run **prediction** later in a fresh kernel after executing only the imports and definition cells, provided `best_model.pkl` is present. Do not fit inside the prediction workflow.

In [ ]:
# train()

In [ ]:
# predict()